In [5]:
# ============================================================
# Mixed Fine-Tuned YOLO26s
# Independent TEST on official LEVIR-Ship TEST split
# Google Colab ONE CELL
# ============================================================

!pip install -q -U ultralytics gdown pyyaml

import shutil
import zipfile
from pathlib import Path

import gdown
import yaml
import torch
from ultralytics import YOLO
from google.colab import drive


# ============================================================
# 1. Google Drive
# ============================================================

drive.mount("/content/drive")


# ============================================================
# 2. Paths
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/LEVIR_Ship_YOLO26"
)

# ------------------------------------------------------------
# Mixed fine-tuned model
#
# 1:1 모델 예시
# ------------------------------------------------------------

MIXED_MODEL = (
    PROJECT_DIR
#    / "Mixed_Finetune"
#    / "LEVIR_Sentinel2_5to1_30epoch"
#    / "weights"
#    / "last.pt"

#    / "YOLO26s_LEVIR_Ship"
#    / "weights"
#    / "last.pt"

    / "Sentinel2_Finetune"
    / "LEVIR_to_Sentinel2"
    / "weights"
    / "best.pt"


)

# 2:1 모델을 테스트하려면 위 대신:
#
# MIXED_MODEL = (
#     PROJECT_DIR
#     / "Mixed_Finetune"
#     / "LEVIR_Sentinel2_2to1"
#     / "weights"
#     / "best.pt"
# )


# ============================================================
# 3. Local LEVIR test directory
# ============================================================

LOCAL_DIR = Path(
    "/content/LEVIR_Ship_Test"
)

DOWNLOAD_DIR = (
    LOCAL_DIR
    / "downloads"
)

EXTRACT_DIR = (
    LOCAL_DIR
    / "extracted"
)

DATASET_DIR = (
    LOCAL_DIR
    / "dataset"
)

DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 4. Official LEVIR-Ship TEST Google Drive ID
# ============================================================

TEST_FILE_ID = (
    "1SRq7hq7glKzzePWqtSlt0C_RVtc1pQQC"
)


# ============================================================
# 5. Download official TEST zip only
# ============================================================

TEST_ZIP = (
    DOWNLOAD_DIR
    / "test.zip"
)


if not TEST_ZIP.exists():

    print("=" * 70)
    print("DOWNLOADING LEVIR TEST SET")
    print("=" * 70)

    gdown.download(
        id=TEST_FILE_ID,
        output=str(TEST_ZIP),
        quiet=False
    )

else:

    print(
        "LEVIR test.zip already exists."
    )


# ============================================================
# 6. Extract TEST
# ============================================================

TEST_EXTRACT = (
    EXTRACT_DIR
    / "test"
)


if not TEST_EXTRACT.exists():

    print()
    print("=" * 70)
    print("EXTRACTING LEVIR TEST SET")
    print("=" * 70)

    TEST_EXTRACT.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        TEST_ZIP,
        "r"
    ) as z:

        z.extractall(
            TEST_EXTRACT
        )

else:

    print(
        "LEVIR test set already extracted."
    )


# ============================================================
# 7. Prepare YOLO test split
# ============================================================

IMAGE_EXTS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".tif",
    ".tiff"
}


image_files = [
    p
    for p in TEST_EXTRACT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in IMAGE_EXTS
]


label_files = [
    p
    for p in TEST_EXTRACT.rglob("*.txt")
    if p.is_file()
]


print()
print("=" * 70)
print("RAW TEST FILES")
print("=" * 70)

print(
    "Images found:",
    len(image_files)
)

print(
    "Labels found:",
    len(label_files)
)


# ============================================================
# 8. Normalize to:
#
# dataset/
#   test/
#     images/
#     labels/
# ============================================================

TEST_IMG_DIR = (
    DATASET_DIR
    / "test"
    / "images"
)

TEST_LBL_DIR = (
    DATASET_DIR
    / "test"
    / "labels"
)


TEST_IMG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEST_LBL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# label lookup by filename stem
label_map = {
    p.stem: p
    for p in label_files
}


copied_images = 0
copied_labels = 0
negative_samples = 0


for img in image_files:

    # --------------------------------------------------------
    # Copy image
    # --------------------------------------------------------

    dst_img = (
        TEST_IMG_DIR
        / img.name
    )

    if not dst_img.exists():

        shutil.copy2(
            img,
            dst_img
        )

    copied_images += 1


    # --------------------------------------------------------
    # Copy corresponding YOLO label
    # --------------------------------------------------------

    src_label = label_map.get(
        img.stem
    )

    dst_label = (
        TEST_LBL_DIR
        / f"{img.stem}.txt"
    )


    if src_label is not None:

        if not dst_label.exists():

            shutil.copy2(
                src_label,
                dst_label
            )

        copied_labels += 1

        if src_label.stat().st_size == 0:

            negative_samples += 1

    else:

        # YOLO negative image
        dst_label.touch(
            exist_ok=True
        )

        negative_samples += 1


print()
print("=" * 70)
print("PREPARED TEST SET")
print("=" * 70)

print(
    "Images:",
    copied_images
)

print(
    "Existing labels:",
    copied_labels
)

print(
    "Negative samples:",
    negative_samples
)


# ============================================================
# 9. Sanity check
# ============================================================

n_img = len([
    p for p in TEST_IMG_DIR.iterdir()
    if p.suffix.lower() in IMAGE_EXTS
])

n_lbl = len(
    list(
        TEST_LBL_DIR.glob("*.txt")
    )
)


print()
print("=" * 70)
print("SANITY CHECK")
print("=" * 70)

print(
    "Test images:",
    n_img
)

print(
    "Test labels:",
    n_lbl
)


assert n_img > 0, (
    "No LEVIR test images found."
)

assert n_img == n_lbl, (
    f"Image / label mismatch: "
    f"{n_img} vs {n_lbl}"
)

assert MIXED_MODEL.exists(), (
    f"Mixed model not found:\n"
    f"{MIXED_MODEL}"
)


# ============================================================
# 10. Create test-only YAML
# ============================================================

YAML_PATH = (
    LOCAL_DIR
    / "levir_test.yaml"
)


data_yaml = {

    "path": str(
        DATASET_DIR
    ),

    # Ultralytics YAML 형식을 위해 train/val도 넣되
    # 실제 평가에는 test만 사용
    "train": "test/images",

    "val": "test/images",

    "test": "test/images",

    "names": {
        0: "ship"
    }
}


with open(
    YAML_PATH,
    "w"
) as f:

    yaml.safe_dump(
        data_yaml,
        f,
        sort_keys=False
    )


print()
print("=" * 70)
print("TEST YAML")
print("=" * 70)

print(
    YAML_PATH.read_text()
)


# ============================================================
# 11. Device
# ============================================================

DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)


print()
print("=" * 70)
print("DEVICE")
print("=" * 70)

print(
    DEVICE
)

if torch.cuda.is_available():

    print(
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 12. Load mixed fine-tuned model
# ============================================================

print()
print("=" * 70)
print("MIXED MODEL")
print("=" * 70)

print(
    MIXED_MODEL
)


model = YOLO(
    str(MIXED_MODEL)
)


# ============================================================
# 13. Independent LEVIR TEST
# ============================================================

print()
print("=" * 70)
print("LEVIR-SHIP INDEPENDENT TEST")
print("=" * 70)


test_result = model.val(

    data=str(
        YAML_PATH
    ),

    split="test",

    imgsz=640,

    batch=16,

    device=DEVICE,

    workers=2,

    plots=True,

    project=str(
        PROJECT_DIR
        / "Mixed_Finetune_Test"
    ),

    name="LEVIR_Test_1to1",

    exist_ok=True
)


# ============================================================
# 14. Print metrics
# ============================================================

print()
print("=" * 70)
print("FINAL LEVIR TEST METRICS")
print("=" * 70)


print(
    f"Precision : "
    f"{test_result.box.mp:.4f}"
)

print(
    f"Recall    : "
    f"{test_result.box.mr:.4f}"
)

print(
    f"mAP50     : "
    f"{test_result.box.map50:.4f}"
)

print(
    f"mAP50-95  : "
    f"{test_result.box.map:.4f}"
)


print()
print("=" * 70)
print("DONE")
print("=" * 70)

print(
    "Model:"
)

print(
    MIXED_MODEL
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
LEVIR test.zip already exists.
LEVIR test set already extracted.

RAW TEST FILES
Images found: 1576
Labels found: 788

PREPARED TEST SET
Images: 1576
Existing labels: 1576
Negative samples: 788

SANITY CHECK
Test images: 788
Test labels: 788

TEST YAML
path: /content/LEVIR_Ship_Test/dataset
train: test/images
val: test/images
test: test/images
names:
  0: ship


DEVICE
0
Tesla T4

MIXED MODEL
/content/drive/MyDrive/LEVIR_Ship_YOLO26/Sentinel2_Finetune/LEVIR_to_Sentinel2/weights/best.pt

LEVIR-SHIP INDEPENDENT TEST
Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1814.7±975.5 MB/s, size: 45.8 KB)
val: Scanning /content/LEVIR_Ship_Test/dataset/test/labels.cache... 788 images, 394 backgrounds, 0 